# RAG Step 1: Document Loading and Chunking
Pipeline: **Load -> Chunk -> Embed -> Store -> Retrieve -> Augment -> Generate -> Evaluate**. This notebook covers Load and Chunk.

In [ ]:
!pip install -q pypdf langchain-text-splitters

## 1. Load documents
We create a small sample knowledge base as text files.

In [ ]:
from pathlib import Path

corpus = {
  "ai_basics.txt": "Machine learning lets computers learn patterns from data instead of following hand-written rules. Supervised learning uses labelled examples, while unsupervised learning finds structure in unlabelled data. Deep learning uses multi-layer neural networks and powers modern vision and language systems.",
  "rag_overview.txt": "Retrieval-Augmented Generation (RAG) combines a retriever with a language model. The retriever finds relevant passages from a knowledge base, and the language model uses them as context to write a grounded answer. RAG reduces hallucinations and lets models use private or up-to-date data without retraining.",
  "vector_db.txt": "A vector database stores embeddings and supports fast nearest-neighbour search. ChromaDB, FAISS and Pinecone are popular choices. Similarity is usually measured with cosine similarity or dot product over normalised vectors.",
  "chunking.txt": "Chunking splits long documents into smaller passages before embedding. Chunks that are too large dilute meaning, while chunks that are too small lose context. A small overlap between chunks helps preserve information across boundaries.",
}

Path("data").mkdir(exist_ok=True)
for name, body in corpus.items():
    Path("data", name).write_text(body)

docs = [{"source": p.name, "text": p.read_text()} for p in Path("data").glob("*.txt")]
print(len(docs), "documents loaded")

In [ ]:
# PDF loading (point to a real file)
# from pypdf import PdfReader
# reader = PdfReader("your_file.pdf")
# text = "\n".join(page.extract_text() for page in reader.pages)

## 2. Fixed-size chunking with overlap (from scratch)

In [ ]:
def chunk_text(text, size=150, overlap=30):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

c = chunk_text(docs[0]["text"])
print(len(c), "chunks")
print(repr(c[0][-30:]), "|", repr(c[1][:30]))

## 3. Recursive splitting (respects sentences)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""])
records = []
for d in docs:
    for i, ch in enumerate(splitter.split_text(d["text"])):
        records.append({"id": f"{d['source']}-{i}", "text": ch, "metadata": {"source": d["source"], "chunk": i}})
print(len(records), "chunks total")
print(records[0])

## 4. Effect of chunk size

In [ ]:
for size in (80, 150, 300):
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 5)
    print(f"chunk_size={size:>3} -> {sum(len(s.split_text(d['text'])) for d in docs)} chunks")

In [ ]:
import json
json.dump(records, open("chunks.json", "w"))  # reused in Step 2

## Takeaways
Good chunks are self-contained, moderately sized and slightly overlapping. Next: embed and store them.